In [ ]:
!pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd

result, _ = reviews("com.duolingo", lang="en", country="us",
                    sort=Sort.NEWEST, count=25000)
raw = pd.DataFrame(result)[["reviewId", "at", "score", "content", "thumbsUpCount"]]
raw = raw.drop_duplicates(subset="content").dropna(subset=["content"])
raw = raw[raw["content"].str.split().str.len() >= 4]

print("rows:", len(raw))
print("date range:", raw["at"].min(), "to", raw["at"].max())
print(raw["score"].value_counts().sort_index())
raw.to_csv("duolingo_reviews_all.csv", index=False)

In [ ]:
neg = raw[raw["score"] <= 2]
mid = raw[raw["score"] == 3]
pos = raw[raw["score"] >= 4]

sample = pd.concat([
    neg.sample(min(len(neg), 600), random_state=42),
    mid.sample(min(len(mid), 300), random_state=42),
    pos.sample(min(len(pos), 1100), random_state=42),
]).sort_values("at").reset_index(drop=True)

sample.insert(0, "id", range(1, len(sample) + 1))
sample.to_csv("duolingo_reviews_sample.csv", index=False)
print(len(sample))
print(sample["score"].value_counts().sort_index())

from google.colab import files
files.download("duolingo_reviews_sample.csv")

In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd

result, _ = reviews(
    "com.duolingo",
    lang="en",
    country="us",
    sort=Sort.NEWEST,
    count=2500,
)

df = pd.DataFrame(result)[["at", "score", "content", "thumbsUpCount"]]
df = df.drop_duplicates(subset="content").dropna(subset=["content"])
df = df[df["content"].str.split().str.len() >= 4]
df = df.head(2000)
df.to_csv("duolingo_reviews.csv", index=False)
print(len(df))
df.head()

In [ ]:
from google.colab import files
files.download("duolingo_reviews.csv")

In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd

result, _ = reviews("com.duolingo", lang="en", country="us",
                    sort=Sort.NEWEST, count=2500)
df = pd.DataFrame(result)[["at", "score", "content", "thumbsUpCount"]]
df = df.drop_duplicates(subset="content").dropna(subset=["content"])
df = df[df["content"].str.split().str.len() >= 4]
df = df.head(2000)
df.to_csv("duolingo_reviews.csv", index=False)

print("rows:", len(df))
print("date range:", df["at"].min(), "to", df["at"].max())
print(df["score"].value_counts().sort_index())

In [ ]:
from google.colab import files
files.download("duolingo_reviews.csv")

In [ ]:
!pip install -q google-genai

In [ ]:
import pandas as pd, json, time, os

MODEL = "gemini-3.1-flash-lite"
df = pd.read_csv("duolingo_reviews_sample.csv")

THEMES = ["Ads", "Pricing and paywall", "Billing and cancellation",
          "Customer support", "Bugs and technical issues",
          "Content and lesson quality", "Streaks and motivation",
          "Learning effectiveness and ease", "Other"]

PROMPT = """You are tagging Google Play reviews of the Duolingo app for a product analysis.
For each review, assign exactly ONE theme and ONE sentiment.

Themes (use these exact labels):
Ads
Pricing and paywall
Billing and cancellation
Customer support
Bugs and technical issues
Content and lesson quality
Streaks and motivation
Learning effectiveness and ease
Other

Rules:
- Choose the theme for the main point of the review. If there are several, pick the one the reviewer stresses most or mentions first.
- Charges, trials, refunds, or cancelling go under Billing and cancellation. Complaints about cost, forced upgrades, or energy/hearts limits go under Pricing and paywall.
- Sentiment is positive, negative, or mixed, based on the review text, not the star rating.
- Use Other only if nothing fits.

Return ONLY a JSON list, one object per review, like:
[{"id": 12, "theme": "Ads", "sentiment": "negative"}]

Reviews:
"""

def tag_chunk(chunk):
    payload = "\n".join(
        json.dumps({"id": int(r.id), "review": r.content[:400]})
        for r in chunk.itertuples())
    resp = client.models.generate_content(
        model=MODEL, contents=PROMPT + payload,
        config={"response_mime_type": "application/json", "temperature": 0})
    rows = json.loads(resp.text)
    return [r for r in rows if r.get("theme") in THEMES
            and r.get("sentiment") in ("positive", "negative", "mixed")]

test = df.head(20)
result = pd.DataFrame(tag_chunk(test))
check = test.merge(result, on="id")

for i, r in enumerate(check.itertuples(), start=1):
    print(f"{i}. [{r.score} star] {r.theme} | {r.sentiment}")
    print(f"   {r.content[:150]}")
    print()

    check[["id", "score", "content", "theme", "sentiment"]].to_csv("test_20.csv", index=False)
from google.colab import files
files.download("test_20_.csv")

In [ ]:
from google.colab import files
files.upload()

In [ ]:
OUT = "tags.csv"
done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame(columns=["id", "theme", "sentiment"])
todo = df[~df["id"].isin(done["id"])]
BATCH = 100

for i in range(0, len(todo), BATCH):
    chunk = todo.iloc[i:i + BATCH]
    try:
        rows = pd.DataFrame(tag_chunk(chunk))
    except Exception as e:
        print("Stopped:", str(e)[:200])
        break
    rows.to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
    print("saved batch, total so far:", len(pd.read_csv(OUT)))
    time.sleep(8)

In [ ]:
tags = pd.read_csv(OUT).drop_duplicates("id")
final = df.merge(tags, on="id", how="left")
print("untagged:", final["theme"].isna().sum())
print(final["theme"].value_counts())
final.to_csv("duolingo_tagged.csv", index=False)

from google.colab import files
files.download("duolingo_tagged.csv")

In [ ]:
print(THEMES)
print("gamification" in PROMPT)

In [ ]:
import os
if os.path.exists("tags.csv"): os.remove("tags.csv")

In [ ]:
import pandas as pd, json, time, os

MODEL = "gemini-3.1-flash-lite"
df = pd.read_csv("duolingo_reviews_sample.csv")

THEMES = ["Ads", "Pricing and paywall", "Billing and cancellation",
          "Customer support", "Bugs and technical issues",
          "Content and lesson quality", "Streaks and gamification",
          "Learning effectiveness and ease", "Other"]

PROMPT = """You are tagging Google Play reviews of the Duolingo app for a product analysis.
For each review, assign exactly ONE theme and ONE sentiment.

Themes (use these exact labels):
Ads
Pricing and paywall
Billing and cancellation
Customer support
Bugs and technical issues
Content and lesson quality
Streaks and gamification
Learning effectiveness and ease
Other

Rules:
- Choose the theme for the main point of the review. If there are several, pick the one the reviewer stresses most or mentions first.
- Charges, trials, refunds, or cancelling go under Billing and cancellation. Complaints about cost, forced upgrades, or energy/hearts limits go under Pricing and paywall.
- Streaks and gamification covers streaks, streak loss, reminders, chests, rewards, leagues, achievements, animations after lessons, social features, and the game or slot machine feel of the app. Use Ads only for actual advertisements.
- If a review says the app does not actually teach well, use Learning effectiveness and ease, even if it mentions streaks.
- Sentiment is positive, negative, or mixed, based on the review text, not the star rating.
- If a review gives a specific reason (for example "easy to use", "fun way to learn", "too many ads"), tag that topic.
- Use Other for reviews with no specific topic, such as generic praise or complaints ("good app", "nice", "worst app", "not good at all"), and for spam or unreadable text.
- Use Other only if nothing else fits.

Return ONLY a JSON list, one object per review, like:
[{"id": 12, "theme": "Ads", "sentiment": "negative"}]

Reviews:
"""

def tag_chunk(chunk):
    payload = "\n".join(
        json.dumps({"id": int(r.id), "review": r.content[:400]})
        for r in chunk.itertuples())
    resp = client.models.generate_content(
        model=MODEL, contents=PROMPT + payload,
        config={"response_mime_type": "application/json", "temperature": 0})
    rows = json.loads(resp.text)
    return [r for r in rows if r.get("theme") in THEMES
            and r.get("sentiment") in ("positive", "negative", "mixed")]

def tag_chunk_retry(chunk, tries=5):
    for attempt in range(tries):
        try:
            return tag_chunk(chunk)
        except Exception as e:
            msg = str(e)
            if "503" in msg or "UNAVAILABLE" in msg:
                wait = 20 * (attempt + 1)
                print(f"Model busy, waiting {wait} seconds (try {attempt + 1} of {tries})")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Model still busy after retries")

print(THEMES)
print("gamification" in PROMPT)
print("no specific topic" in PROMPT)

In [ ]:
import os
if os.path.exists("tags.csv"): os.remove("tags.csv")

In [ ]:
import pandas as pd, json, time, os

MODEL = "gemini-3.1-flash-lite"
df = pd.read_csv("duolingo_reviews_sample.csv")

THEMES = ["Ads", "Pricing and paywall", "Billing and cancellation",
          "Customer support", "Bugs and technical issues",
          "Content and lesson quality", "Streaks and gamification",
          "Learning effectiveness and ease", "Other"]

PROMPT = """You are tagging Google Play reviews of the Duolingo app for a product analysis.
For each review, assign exactly ONE theme and ONE sentiment.

Themes (use these exact labels):
Ads
Pricing and paywall
Billing and cancellation
Customer support
Bugs and technical issues
Content and lesson quality
Streaks and gamification
Learning effectiveness and ease
Other

Rules:
- Choose the theme for the main point of the review. If there are several, pick the one the reviewer stresses most or mentions first.
- Charges, trials, refunds, or cancelling go under Billing and cancellation. Complaints about cost, forced upgrades, or energy/hearts limits go under Pricing and paywall.
- Streaks and gamification covers streaks, streak loss, reminders, chests, rewards, leagues, achievements, animations after lessons, social features, and the game or slot machine feel of the app. Use Ads only for actual advertisements.
- If a review says the app does not actually teach well, use Learning effectiveness and ease, even if it mentions streaks.
- If a review gives a specific reason (for example "easy to use", "fun way to learn", "too many ads"), tag that topic.
- Use Other for reviews with no specific topic, such as generic praise or complaints ("good app", "nice", "worst app", "not good at all"), and for spam or unreadable text.
- Use Other only if nothing else fits.
- Sentiment is positive, negative, or mixed, based on the review text, not the star rating.

Return ONLY a JSON list, one object per review, like:
[{"id": 12, "theme": "Ads", "sentiment": "negative"}]

Reviews:
"""

def tag_chunk(chunk):
    payload = "\n".join(
        json.dumps({"id": int(r.id), "review": r.content[:400]})
        for r in chunk.itertuples())
    resp = client.models.generate_content(
        model=MODEL, contents=PROMPT + payload,
        config={"response_mime_type": "application/json", "temperature": 0})
    rows = json.loads(resp.text)
    return [r for r in rows if r.get("theme") in THEMES
            and r.get("sentiment") in ("positive", "negative", "mixed")]

def tag_chunk_retry(chunk, tries=5):
    for attempt in range(tries):
        try:
            return tag_chunk(chunk)
        except Exception as e:
            msg = str(e)
            if "503" in msg or "UNAVAILABLE" in msg:
                wait = 20 * (attempt + 1)
                print(f"Model busy, waiting {wait} seconds (try {attempt + 1} of {tries})")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Model still busy after retries")

print("new prompt live:", "no specific topic" in PROMPT)
print("old tags deleted:", not os.path.exists("tags.csv"))

In [ ]:
OUT = "tags.csv"
done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame(columns=["id", "theme", "sentiment"])
todo = df[~df["id"].isin(done["id"])]
BATCH = 100

for i in range(0, len(todo), BATCH):
    chunk = todo.iloc[i:i + BATCH]
    try:
        rows = pd.DataFrame(tag_chunk_retry(chunk))
    except Exception as e:
        print("Stopped:", str(e)[:200])
        break
    rows.to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
    print("saved batch, total so far:", len(pd.read_csv(OUT)))
    time.sleep(8)

In [ ]:
tags = pd.read_csv("tags.csv").drop_duplicates("id")
final = df.merge(tags, on="id", how="left")
print("untagged (will be dropped):", final["theme"].isna().sum())
final = final.dropna(subset=["theme"])
print("tagged:", len(final))
print(final["theme"].value_counts())
final.to_csv("duolingo_tagged.csv", index=False)

from google.colab import files
files.download("duolingo_tagged.csv")

In [ ]:
from google.colab import files
mine = pd.read_csv(list(files.upload().keys())[0])

cmp = mine[["id", "my_theme", "my_sentiment"]].merge(
    final[["id", "theme", "sentiment", "score", "content"]], on="id")

lower_map = {t.lower(): t for t in THEMES}
cmp["my_theme"] = cmp["my_theme"].fillna("").str.strip()
cmp["my_theme"] = cmp["my_theme"].str.lower().map(lower_map).fillna(cmp["my_theme"])
cmp["my_sentiment"] = cmp["my_sentiment"].fillna("").str.strip().str.lower()

cmp["theme_ok"] = cmp["my_theme"] == cmp["theme"]
cmp["sent_ok"] = cmp["my_sentiment"] == cmp["sentiment"]

print("Rows scored:", len(cmp))
print("Theme accuracy:", round(cmp["theme_ok"].mean(), 2))
print("Sentiment accuracy:", round(cmp["sent_ok"].mean(), 2))
print()
for r in cmp[~cmp["theme_ok"]].itertuples():
    print(f"Me: {r.my_theme} | Model: {r.theme}")
    print(f"   {r.content[:140]}")
    print()

In [ ]:
test = df.head(20)
result = pd.DataFrame(tag_chunk_retry(test))
check = test.merge(result, on="id")

for i, r in enumerate(check.itertuples(), start=1):
    print(f"{i}. [{r.score} star] {r.theme} | {r.sentiment}")
    print(f"   {r.content[:150]}")
    print()
check[["id", "score", "content", "theme", "sentiment"]].to_csv("test_20.csv", index=False)
from google.colab import files
files.download("test_20.csv")

In [ ]:
OUT = "tags.csv"
done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame(columns=["id", "theme", "sentiment"])
todo = df[~df["id"].isin(done["id"])]
BATCH = 100

for i in range(0, len(todo), BATCH):
    chunk = todo.iloc[i:i + BATCH]
    try:
        rows = pd.DataFrame(tag_chunk_retry(chunk))
    except Exception as e:
        print("Stopped:", str(e)[:200])
        break
    rows.to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
    print("saved batch, total so far:", len(pd.read_csv(OUT)))
    time.sleep(8)

In [ ]:
tags = pd.read_csv(OUT).drop_duplicates("id")
final = df.merge(tags, on="id", how="left")
print("untagged:", final["theme"].isna().sum())
print(final["theme"].value_counts())
final.to_csv("duolingo_tagged.csv", index=False)

from google.colab import files
files.download("duolingo_tagged.csv")

In [ ]:
missing = final[final["theme"].isna()]
for r in missing.itertuples():
    print(r.id, "|", r.score, "star |", r.content[:150])

In [ ]:
words = "charged|charge|refund|cancel|trial|billed|subscription|auto"
p = final[(final["theme"] == "Pricing and paywall") &
          (final["content"].str.contains(words, case=False, na=False))]
print(len(p), "of", (final["theme"] == "Pricing and paywall").sum())

for r in p.head(25).itertuples():
    print(f"[{r.score} star] {r.content[:140]}")
    print()

In [ ]:
final = final.dropna(subset=["theme"])
print(len(final))
final.to_csv("duolingo_tagged.csv", index=False)

from google.colab import files
files.download("duolingo_tagged.csv")

In [ ]:
check = final.dropna(subset=["theme"]).sample(50, random_state=7)
check[["id", "score", "content"]].assign(my_theme="", my_sentiment="").to_csv("accuracy_check.csv", index=False)

from google.colab import files
files.download("accuracy_check.csv")

In [ ]:
from google.colab import files
mine = pd.read_csv(list(files.upload().keys())[0])

cmp = mine[["id", "my_theme", "my_sentiment"]].merge(
    final[["id", "theme", "sentiment", "score", "content"]], on="id")
cmp["theme_ok"] = cmp["my_theme"].str.strip() == cmp["theme"]
cmp["sent_ok"] = cmp["my_sentiment"].str.strip() == cmp["sentiment"]

print("Rows scored:", len(cmp))
print("Theme accuracy:", round(cmp["theme_ok"].mean(), 2))
print("Sentiment accuracy:", round(cmp["sent_ok"].mean(), 2))
print()
print(pd.crosstab(cmp["my_theme"], cmp["theme"]))
print()
for r in cmp[~cmp["theme_ok"]].itertuples():
    print(f"Me: {r.my_theme} | Model: {r.theme}")
    print(f"   {r.content[:140]}")
    print()

In [ ]:
bad = set(cmp["my_theme"].dropna().str.strip()) - set(THEMES)
print("Labels of mine that don't match a theme name:", bad)
print()

pairs = (cmp[~cmp["theme_ok"]]
         .groupby(["my_theme", "theme"]).size()
         .sort_values(ascending=False))
print("Disagreements (my label -> model label):")
print(pairs.to_string())
print()

for r in cmp[~cmp["theme_ok"]].itertuples():
    print(f"Me: {r.my_theme} | Model: {r.theme} | {r.score} star")
    print(f"   {r.content[:140]}")
    print()

In [ ]:
cmp["my_theme"] = cmp["my_theme"].fillna("").str.strip()
lower_map = {t.lower(): t for t in THEMES}
cmp["my_theme"] = cmp["my_theme"].str.lower().map(lower_map).fillna(cmp["my_theme"])
cmp["theme_ok"] = cmp["my_theme"] == cmp["theme"]
print("Theme accuracy:", round(cmp["theme_ok"].mean(), 2))

In [ ]:
prev = set(mine["id"])
fresh = final[~final["id"].isin(prev)].sample(30, random_state=11)
fresh[["id", "score", "content"]].assign(my_theme="", my_sentiment="").to_csv("accuracy_check_2.csv", index=False)

from google.colab import files
files.download("accuracy_check_2.csv")

In [ ]:
import pandas as pd
from google.colab import files

THEMES = ["Ads", "Pricing and paywall", "Billing and cancellation",
          "Customer support", "Bugs and technical issues",
          "Content and lesson quality", "Streaks and gamification",
          "Learning effectiveness and ease", "Other"]

print("UPLOAD 1 of 2: your tagged data (duolingo_tagged.csv)")
final = pd.read_csv(list(files.upload().keys())[0])
print("tagged rows loaded:", len(final))

print("UPLOAD 2 of 2: your labeled fresh 30 file from Google Sheets")
m2 = pd.read_csv(list(files.upload().keys())[0])

c2 = m2[["id", "my_theme", "my_sentiment"]].merge(
    final[["id", "theme", "sentiment", "score", "content"]], on="id")

lower_map = {t.lower(): t for t in THEMES}
c2["my_theme"] = c2["my_theme"].fillna("").str.strip()
c2["my_theme"] = c2["my_theme"].str.lower().map(lower_map).fillna(c2["my_theme"])
c2["my_sentiment"] = c2["my_sentiment"].fillna("").str.strip().str.lower()

c2["theme_ok"] = c2["my_theme"] == c2["theme"]
c2["sent_ok"] = c2["my_sentiment"] == c2["sentiment"]

print()
print("Rows scored:", len(c2))
print("Theme accuracy:", round(c2["theme_ok"].mean(), 2))
print("Sentiment accuracy:", round(c2["sent_ok"].mean(), 2))
print()
for r in c2[~c2["theme_ok"]].itertuples():
    print(f"Me: {r.my_theme} | Model: {r.theme}")
    print(f"   {r.content[:130]}")
    print()